In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''

from dotenv import load_dotenv
from collections.abc import Sequence
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output

import flax.jax_utils as flax_utils
import flax.linen as nn
import grain.python as grain
import jax
import numpy as np
from absl import logging
from connectomics.jax import checkpoint, training
from etils import epath
from orbax import checkpoint as ocp

import zapbench.models.util as model_util
from zapbench.ts_forecasting import heads, input_pipeline, train
from zapbench.ts_forecasting.configs import infer, mean, linear, timemix, tsmixer, tide

import matplotlib.pyplot as plt
import scienceplots
plt.style.use(['science'])

load_dotenv()
PATH = os.getenv("ROOT_PATH")
LOG_PATH = os.getenv("LOG_PATH")

def format_ax(ax):
  for spine in ax.spines.values():
    spine.set_linewidth(1.2)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.spines['bottom'].set_visible(False)
  ax.tick_params(which='minor', length=0)
  ax.tick_params(axis='both', labelsize=12)
  # Set all spines invisible
  for spine in ax.spines.values():
    spine.set_visible(False)
  # Hide all ticks and tick labels
  ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
  ax.tick_params(axis='y', which='both', left=False, right=False, direction="out", width=1.2)

In [ ]:
config = mean.get_config("dataset_name=subject_17,timesteps_input=4")
print(config.train_specs[0]['timeseries']['transform']['output'][0]['index_array']) #2045 5024

In [ ]:
2466-36, 2430-2045

In [ ]:
from zapbench.ts_forecasting import data_source

train_source = data_source.ConcatenatedTensorStoreTimeSeries(*[
    input_pipeline._build_merged_data_source(
        series=series,
        timesteps_input=config.timesteps_input,
        timesteps_output=config.timesteps_output,
        prefetch=config.prefetch,
        sequential=config.sequential_data_source,
    )
    for series in config.train_specs
])

In [ ]:
train_source[0]['timeseries_input'][0]

In [ ]:
import tensorstore as ts

x = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file://{PATH}/ts_files/subject_17_traces.zarr'
}).result().read().result()

In [ ]:
x.shape

In [ ]:
print(np.all(x[2431]==train_source[386]['timeseries_input'][0]))
print(np.all(x[2466]==train_source[386]['timeseries_output'][-1]))

In [ ]:
print(np.all(x[4855]==train_source[387]['timeseries_input'][0]))

In [ ]:
transform = config.train_specs[0]['timeseries']['transform']

x = [t[0] for t in transform['output'][0]['index_array']]

In [ ]:
x

In [ ]:
from zapbench import data_utils
# data_utils.get_condition_bounds(3) # only for original zapbench
data_utils.get_condition_intervals(0, dataset_name="subject_17")
splits = ['train', 'val', 'test', 'test_holdout']
for split in splits:
  print(split)
  # print(data_utils.adjust_condition_bounds_for_split(split, 3079, 3734, 4))
  window_size = data_utils.calculate_window_size(4)
  valid_timesteps = data_utils.build_valid_timesteps(((2045, 2467), (4855, 5277)), 32)
  print(data_utils.adjust_valid_timesteps_for_split(valid_timesteps, split, 4)[0], data_utils.adjust_valid_timesteps_for_split(valid_timesteps, split, 4)[-1])